In [ ]:
# this is my working code: it clones the sample list, saves the metrics and collects the config, build and test lines
# archiving folder is on extranl drive
import pandas as pd
import os
import subprocess
import shutil
import glob
import random
from pathlib import Path
from dotenv import load_dotenv, set_key
import requests

# === CONFIGURATION ===
MAX_PROJECTS = 1796
RANDOM_SEED = 42
NUM_SAMPLES_TO_KEEP = 150

#read the start number 
# Load .env variables
load_dotenv('All_tokens.env')
START_NUMBER = int(os.getenv("START_NUMBER", "1"))


# === LOAD GITHUB TOKEN ===
load_dotenv('All_tokens.env')
GITHUB_TOKEN = os.getenv('GITHUB_TOKEN')
if not GITHUB_TOKEN:
    raise ValueError("❌ GitHub token not found in All_tokens.env")

headers = {'Authorization': f'token {GITHUB_TOKEN}'}

# === PATHS ===
csv_path = r"C:\GitHub\Android-Mobile-Apps\8.1-Project_GitHub_URLs.csv"
base_dir = Path(r"E:\Android Mobile Project\AndroidProjects")
clone_dir = base_dir / "Cloned repos"
cloned_sample_dir = base_dir / "Cloned_Sample"
yml_output_dir = base_dir / "Config Files"
commits_dir = base_dir / "Commits"
build_info_dir = base_dir / "BuildInfo"
metadata_csv = base_dir / "8.2-Project_Metadata.csv"

# === CLEAN OLD DATA ===
for path in [clone_dir, cloned_sample_dir, yml_output_dir, commits_dir, build_info_dir]:
    if path.exists():
        shutil.rmtree(path)
    path.mkdir(parents=True, exist_ok=True)

# === LOAD AND CLEAN CSV ===
df = pd.read_csv(csv_path)
df.columns = df.columns.str.strip().str.lower()
df = df[df['github_url'].notna()]
df['github_url'] = df['github_url'].astype(str).str.strip()
df = df[df['github_url'].str.startswith("https://")].reset_index(drop=True)

# === SELECT RANDOM SAMPLE FOR FULL CLONE KEEP ===
random.seed(RANDOM_SEED)
sample_indices_to_keep = set(random.sample(range(len(df)), min(NUM_SAMPLES_TO_KEEP, len(df))))

# === PREP METADATA COLLECTION ===
metadata_rows = []

# === PROCESS EACH REPO ===
#for i, url in enumerate(df['github_url'], 1):
for i in range(START_NUMBER - 1,len(df)):
    url = df.loc[i,'github_url']
    if i > MAX_PROJECTS:
        break

    parts = url.split('/')
    if len(parts) < 5:
        continue
    username, project = parts[-2], parts[-1].replace('.git', '')
    repo_name = f"{username}.{project}"
    repo_path = clone_dir / repo_name

    print(f"\n🔍 [{i}/{len(df)}] Processing {repo_name}...")

    # --- Shallow Clone ---
    try:
        subprocess.run(
            ['git', 'clone', '--depth', '1', url, str(repo_path)],
            stdout=subprocess.DEVNULL,
            stderr=subprocess.DEVNULL
        )
    except subprocess.TimeoutExpired:
        print(f"⏱️ Timeout while cloning {repo_name}, skipping...")
        continue
    print("✅ Clone complete")

    # --- Extract .yml/.yaml Files ---
    for root, _, files in os.walk(repo_path):
        for file in files:
            if file.endswith(('.yml', '.yaml')):
                full_path = Path(root) / file
                rel_path = full_path.relative_to(repo_path)
                safe_name = f"{repo_name}.{str(rel_path).replace(os.sep, '_')}"
                shutil.copy2(full_path, yml_output_dir / safe_name)
    print("📄 Config files extracted")

    # --- Extract build.gradle test lines ---
    info_path = build_info_dir / f"{repo_name}_build_info.txt"
    with open(info_path, 'w', encoding='utf-8') as out_file:
        for gradle_file in glob.glob(str(repo_path / '**/*.gradle*'), recursive=True):
            try:
                with open(gradle_file, 'r', encoding='utf-8', errors='ignore') as f:
                    lines = f.readlines()
                    test_lines = [line for line in lines if 'test' in line.lower()]
                    if test_lines:
                        out_file.write(f"\n--- {gradle_file} ---\n")
                        out_file.writelines(test_lines)
            except Exception:
                continue
    print("🛠️ Build info extracted")

    # --- Save Commit Log ---
    with open(commits_dir / f"{repo_name}_commits.txt", 'w', encoding='utf-8') as f:
        subprocess.run(['git', 'log', '--pretty=format:%h | %an | %ad | %s'],
                       cwd=repo_path, stdout=f, stderr=subprocess.DEVNULL)
    print("📜 Commits saved")

    # --- Save Contributors via GitHub API ---
    try:
        api_url = f"https://api.github.com/repos/{username}/{project}/contributors"
        r = requests.get(api_url, headers=headers, timeout=30)
        if r.ok:
            contributors_data = r.json()
            contrib_path = commits_dir / f"{repo_name}_contributors.txt"
            with open(contrib_path, 'w', encoding='utf-8') as f:
                for contributor in contributors_data:
                    f.write(f"{contributor['contributions']:>4} | {contributor['login']}\n")
            print("👥 Contributors saved via GitHub API")
        else:
            print(f"⚠️ GitHub API error ({r.status_code}) for contributors of {repo_name}")
    except Exception as e:
        print(f"⚠️ Error retrieving contributors for {repo_name}: {e}")

    # --- GitHub API Metadata (Full Metrics) ---
    api_url = f"https://api.github.com/repos/{username}/{project}"
    try:
        r = requests.get(api_url, headers=headers, timeout=30)
        if r.ok:
            data = r.json()

            # Contributor count
            contributors_url = f"https://api.github.com/repos/{username}/{project}/contributors"
            contrib_count = 0
            try:
                contrib_resp = requests.get(contributors_url, headers=headers, timeout=15)
                if contrib_resp.ok:
                    contrib_count = len(contrib_resp.json())
            except Exception:
                pass

            # Pull request count
            pull_requests = 0
            try:
                pr_url = f"https://api.github.com/repos/{username}/{project}/pulls?state=all&per_page=1"
                pr_resp = requests.get(pr_url, headers=headers)
                if 'Link' in pr_resp.headers:
                    pr_last_page = pr_resp.headers['Link'].split('page=')[-1].split('>')[0]
                    pull_requests = int(pr_last_page)
                else:
                    pull_requests = len(pr_resp.json())
            except Exception:
                pass

            # Commit count
            commit_count = 0
            try:
                commits_url = f"https://api.github.com/repos/{username}/{project}/commits?per_page=1"
                commits_resp = requests.get(commits_url, headers=headers)
                if 'Link' in commits_resp.headers:
                    last_page = commits_resp.headers['Link'].split('page=')[-1].split('>')[0]
                    commit_count = int(last_page)
                else:
                    commit_count = len(commits_resp.json())
            except Exception:
                pass

            metadata_rows.append({
                'repo_name': repo_name,
                'full_name': data.get('full_name'),
                'description': data.get('description'),
                'language': data.get('language'),
                'license': data.get('license', {}).get('name') if data.get('license') else None,
                'created_at': data.get('created_at'),
                'updated_at': data.get('updated_at'),
                'last_commit_date': data.get('pushed_at'),
                'stars': data.get('stargazers_count'),
                'forks': data.get('forks_count'),
                'watchers': data.get('watchers_count'),
                'open_issues': data.get('open_issues_count'),
                'contributors': contrib_count,
                'pull_requests': pull_requests,
                'commits': commit_count,
                'size': data.get('size')
            })
            print("📊 Full metrics retrieved")
        else:
            print(f"⚠️ Metadata fetch failed for {repo_name}")
    except Exception as e:
        print(f"⚠️ Exception retrieving metadata for {repo_name}: {e}")

    # --- Move Sampled Repos to Cloned_Sample Folder ---
        # --- Move or Delete Repos and Clean Cloned Folder ---
    try:
        if i - 1 in sample_indices_to_keep:
            # Move to Cloned_Sample
            dest_path = cloned_sample_dir / repo_path.name
            if dest_path.exists():
                shutil.rmtree(dest_path, ignore_errors=True)
            shutil.move(str(repo_path), str(dest_path))
            print(f"📦 Sample repo moved to: {dest_path}")
        else:
            # Delete non-sample repo
            if repo_path.exists():
                shutil.rmtree(repo_path, ignore_errors=True)
                print(f"🗑️ Non-sample repo deleted: {repo_path.name}")
    except Exception as e:
        print(f"❌ Error moving or deleting repo {repo_name}: {e}")
    
    # Update .env with new START_NUMBER (i + 2 since it's 1-indexed)
    set_key('All_tokens.env', 'START_NUMBER', str(i + 2))


# === SAVE METADATA CSV ===
if metadata_rows:
    pd.DataFrame(metadata_rows).to_csv(metadata_csv, index=False)
    print(f"\n✅ Saved metadata for {len(metadata_rows)} projects")

print("\n🏁 Finished processing selected projects.")


# Save next start index to .env
set_key('All_tokens.env', 'START_NUMBER', str(i + 2))




🔍 [1/1796] Processing AChep.15puzzle...
✅ Clone complete
📄 Config files extracted
🛠️ Build info extracted
📜 Commits saved
👥 Contributors saved via GitHub API
📊 Full metrics retrieved
🗑️ Non-sample repo deleted: AChep.15puzzle

🔍 [2/1796] Processing hapramp.1Rramp-Android...
✅ Clone complete
📄 Config files extracted
🛠️ Build info extracted
📜 Commits saved
👥 Contributors saved via GitHub API
📊 Full metrics retrieved
🗑️ Non-sample repo deleted: hapramp.1Rramp-Android

🔍 [3/1796] Processing CSID-DGU.2021-1-OSSP2-Barcode-8...
✅ Clone complete
📄 Config files extracted
🛠️ Build info extracted
📜 Commits saved
👥 Contributors saved via GitHub API
📊 Full metrics retrieved
🗑️ Non-sample repo deleted: CSID-DGU.2021-1-OSSP2-Barcode-8

🔍 [4/1796] Processing pknu-wap.2022_2_WAP_APP_TEAM1...
✅ Clone complete
📄 Config files extracted
🛠️ Build info extracted
📜 Commits saved
👥 Contributors saved via GitHub API
📊 Full metrics retrieved
🗑️ Non-sample repo deleted: pknu-wap.2022_2_WAP_APP_TEAM1

🔍 [5/1796] 

FileNotFoundError: [Errno 2] No such file or directory: 'E:\\Android Mobile Project\\AndroidProjects\\BuildInfo\\rozPierog.Cofi_build_info.txt'